In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, auc, roc_curve
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

# Load the feature matrix
wards = gpd.read_file("./Data/features_v2.gpkg")

# Drop geometry and metadata for modelling — keep only numeric features and target
feature_cols = [
    'pop2009',
    'rain_cumulative_mm',
    'rain_max_daily_mm',
    'rain_preflood_7d_mm',
    'elevation_mean_m',
    'elevation_min_m',
    'slope_mean_deg'
]

X = wards[feature_cols]
y = wards['flooded']

print(f"Feature matrix shape : {X.shape}")
print(f"Class distribution   :\n{y.value_counts(normalize=True)}")
print(f"\nMissing values:\n{X.isnull().sum()}")

Feature matrix shape : (1450, 7)
Class distribution   :
0    0.788276
1    0.211724
Name: flooded, dtype: float64

Missing values:
pop2009                0
rain_cumulative_mm     0
rain_max_daily_mm      0
rain_preflood_7d_mm    0
elevation_mean_m       0
elevation_min_m        0
slope_mean_deg         0
dtype: int64


In [2]:
# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.head()

,pop2009,rain_cumulative_mm,rain_max_daily_mm,rain_preflood_7d_mm,elevation_mean_m,elevation_min_m,slope_mean_deg
1112,52777.0,0.000000,0.000000,0.000000,34.313187,1.0,4.642516
1298,46351.0,0.000000,0.000000,0.000000,1532.464098,1511.0,1.348357
836,55406.0,371.738520,37.046970,62.983926,1762.939130,1701.0,2.628858
1115,18806.0,0.000000,0.000000,0.000000,18.622155,1.0,1.981015
305,30960.0,138.300461,25.247042,11.300949,1129.965852,853.0,3.620829


In [8]:
# Build a simple XGBoost model
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91       229
           1       0.73      0.52      0.61        61

    accuracy                           0.86       290
   macro avg       0.80      0.74      0.76       290
weighted avg       0.85      0.86      0.85       290



In [10]:
# Handle class imbalance using SMOTE
from imblearn.over_sampling import SMOTE

# Instantiate SMOTE
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Train the model on the SMOTE data
model.fit(X_train_smote, y_train_smote)

# Predict on the test set
y_pred_smote = model.predict(X_test)

# Evaluate the model
print("Classification Report after SMOTE:")
print(classification_report(y_test, y_pred_smote))


c:\Users\Admin\anaconda3\envs\learn-env\lib\site-packages\xgboost\data.py:173: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index


Classification Report after SMOTE:
              precision    recall  f1-score   support

           0       0.91      0.86      0.89       229
           1       0.57      0.69      0.62        61

    accuracy                           0.82       290
   macro avg       0.74      0.77      0.75       290
weighted avg       0.84      0.82      0.83       290

